# Specimen ecotyping from spatial cell graphs

Read curated cell annotations and molecular-spatial features, learn specimen representations by graph contrastive learning, and identify specimen ecotypes with Leiden clustering.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/01_domain_discovery").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "domain_reference_v2")
OUT = ROOT / "outputs/domain_discovery" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
SEED = 42
INPUT_H5AD = OUT / "03_domains.h5ad"
SAMPLE_KEY = "sample"
FEATURE_KEY = "X_cellcharter"
CELL_NEIGHBORS = 7
HIDDEN_DIM = 128
PROJECTION_DIM = 64
EPOCHS = 200
BATCH_SIZE = 8
LEARNING_RATE = 0.001
SPECIMEN_NEIGHBORS = 30
LEIDEN_RESOLUTION = 0.3
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.spatial.ecotyping import (
    build_sample_graphs,
    cluster_ecotypes,
    projected_embeddings,
    train_ecotyper,
)

Read the real annotated cell object produced by domain discovery.

In [ ]:
data = ad.read_h5ad(INPUT_H5AD)

## 2. Construct specimen graphs

Remove low-quality cells and construct a seven-neighbour graph separately within each eligible specimen. Molecular-spatial features serve as node attributes.

Remove low-quality cells and build a nearest-neighbour graph independently for each specimen.

In [ ]:
(graphs, sample_names) = build_sample_graphs(
    data, SAMPLE_KEY, FEATURE_KEY, CELL_NEIGHBORS
)

## 3. Train the graph contrastive encoder

Use two stochastic graph views and the full two-view NT-Xent objective to learn specimen-level representations.

Two graph convolutions encode each specimen; global mean pooling and a projection head provide the contrastive representation.

Generate two graph views by dropping edges and masking the same feature channels across nodes.

Contrast two views of each specimen against all other views, excluding only self-similarities.

Optimize the graph encoder on paired specimen views and retain the training-loss history.

In [ ]:
(model, history) = train_ecotyper(
    graphs,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    projection_dim=PROJECTION_DIM,
    learning_rate=LEARNING_RATE,
    device=DEVICE,
)

## 4. Extract specimen vectors and identify ecotypes

Use projected vectors, not pooled encoder vectors. Standardize and L2-normalize them before cosine-neighbour Leiden clustering.

Extract the projection vectors used for spatial-transcriptomic ecotyping.

Standardize and L2-normalize specimen vectors, then partition their cosine-neighbour graph with Leiden.

In [ ]:
vectors = projected_embeddings(model, graphs, DEVICE)
partition = cluster_ecotypes(
    vectors,
    sample_names,
    neighbors=SPECIMEN_NEIGHBORS,
    resolution=LEIDEN_RESOLUTION,
    seed=SEED,
)

## 5. Save specimen labels, vectors and model weights

Save embeddings, ecotype assignments, training losses and the fitted model; cluster indices require biological interpretation.

In [ ]:
table = pd.DataFrame(vectors, index=pd.Index(sample_names, name=SAMPLE_KEY)).add_prefix(
    "projection_"
)
table["ecotype"] = partition.obs["ecotype"].astype(str).to_numpy()
table.to_csv(require_new(OUT / "05_sample_graph_embeddings.csv"))
pd.DataFrame(history).to_csv(require_new(OUT / "05_training_history.csv"), index=False)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": graphs[0].num_node_features,
        "hidden_dim": HIDDEN_DIM,
        "projection_dim": PROJECTION_DIM,
        "method": "Figure3_GCN_full_NTXent_Leiden",
    },
    require_new(OUT / "05_ecotyper.pt"),
)
assert np.isfinite(vectors).all()
display(table.iloc[:, [0, 1, -1]])